# ProtoFormer — PASCAL-5i fold 0 replication
1. setup
2. data
3. train
4. test
5. save


In [ ]:
# 1
import torch
import os, sys, json, subprocess, shutil, time, tarfile, urllib.request, zipfile
from pathlib import Path

DRIVE_FOLDER = "ProtoFormer_out"  # ganti folder di MyDrive
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    OUT = Path(f"/content/drive/MyDrive/{DRIVE_FOLDER}")
except Exception as e:
    print("drive mount gagal, pakai lokal:", e)
    OUT = Path("/content/out")
OUT.mkdir(parents=True, exist_ok=True)

REPO = "/content/ProtoFormer"
GIT_URL = "https://github.com/leilei2018/ProtoFormer.git"  # ganti punyamu
BRANCH = "main"
DATA = Path("/content/data")
SAM2_CKPT = "/content/sam2_hiera_l.pth"
SAM2_URL = ""  # isi link SAM2 resmi punyamu
SAM2_DRIVE_ID = ""  # atau isi Drive file id

CFG = dict(
    benchmark="pascal", fold=0,
    shots=[1, 5],
    bsz=32, lr=1e-3, niter=60, nworker=4,
    reduce_dim=64,
    amp="auto", logpath="pascal_fold0", patience=20, val_interval=3,
)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print("OUT:", OUT)
print(CFG)


In [ ]:
# 2
# repo
if not Path(REPO).exists():
    subprocess.run(["git", "clone", "-b", BRANCH, GIT_URL, REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "reset", "--hard", f"origin/{BRANCH}"], check=True)
    subprocess.run(["git", "-C", REPO, "clean", "-fd"], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# VOC2012 - pakai https, urllib default UA ditolak 403
voc = DATA / "VOC2012"
if not (voc / "JPEGImages").exists():
    subprocess.run(["wget", "-q", "-O", "/content/VOC.tar",
                    "https://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar"], check=True)
    with tarfile.open("/content/VOC.tar") as t:
        t.extractall("/content", filter="data")
    shutil.move("/content/VOCdevkit/VOC2012", str(DATA / "VOC2012"))

# SegmentationClassAug (Drive id dari README)
aug = DATA / "VOC2012/SegmentationClassAug"
if not aug.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "gdown"],
        check=False)
    subprocess.run(
        [sys.executable, "-m", "gdown", "--fuzzy", "1UvTarXD2jUVFpdHT9n-VG9zLZk4U3ikY",
         "-O", "/content/aug.zip"], check=False)
    if Path("/content/aug.zip").exists():
        with zipfile.ZipFile("/content/aug.zip") as z:
            z.extractall("/content/aug", filter="data")
        src = next(Path("/content/aug").rglob("SegmentationClassAug"))
        shutil.move(str(src), str(aug))

# SAM2 hiera_l
if not Path(SAM2_CKPT).exists():
    if SAM2_URL:
        subprocess.run(["wget", "-O", SAM2_CKPT, SAM2_URL], check=True)
    elif SAM2_DRIVE_ID:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=False)
        subprocess.run([sys.executable, "-m", "gdown", "--fuzzy", SAM2_DRIVE_ID,
                        "-O", SAM2_CKPT], check=True)
    elif Path(OUT / "sam2_hiera_l.pth").exists():
        shutil.copy(OUT / "sam2_hiera_l.pth", SAM2_CKPT)
    else:
        raise RuntimeError("SAM2 checkpoint belum ada: isi SAM2_URL / SAM2_DRIVE_ID di cell 1, atau taruh sam2_hiera_l.pth di OUT (Drive).")
assert Path(SAM2_CKPT).exists()
print("SAM2 OK", Path(SAM2_CKPT).stat().st_size // 1024 // 1024, "MB")
print("OK", list(voc.iterdir())[:3])


In [ ]:
# 3
import sys
sys.path.insert(0, f"{REPO}/model")

import torch
from common.amp import get_device, resolve_amp_dtype, make_scaler, amp_context
from common.logger import Logger, AverageMeter
from common.evaluation import Evaluator
from common import utils
from data.dataset import FSSDataset
from model.fewshot import FewShotSeg as ProtoFormer
import torch.optim as optim

def train_epoch(epoch, model, dl, opt, training, amp_dtype, scaler):
    utils.fix_randseed(None) if training else utils.fix_randseed(0)
    model.train() if training else model.eval()
    m = AverageMeter(dl.dataset)
    for i, b in enumerate(dl):
        b = utils.to_cuda(b)
        if training:
            with amp_context(amp_dtype):
                out = model(b["query_img"], b["support_imgs"], b["support_masks"])
                loss = model.compute_loss(out, b["query_mask"])
            opt.zero_grad()
            if scaler is not None:
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss.backward(); opt.step()
        else:
            with torch.no_grad(), amp_context(amp_dtype):
                out = model.predict_mask(b)
        pred = (out > 0.5).float().squeeze(1)
        inter, union = Evaluator.classify_prediction(pred, b)
        m.update(inter, union, b["class_id"], loss.detach().clone() if training else None)
        m.write_process(i, len(dl), epoch, write_batch_idx=100)
    m.write_result("Training" if training else "Validation", epoch)
    return utils.mean(m.loss_buf), *m.compute_iou()

device = get_device()
amp = resolve_amp_dtype(device, CFG["amp"])
scaler = make_scaler(amp)
print(device, amp)
Evaluator.initialize()
FSSDataset.initialize(img_size=473, datapath=str(DATA))

for shot in CFG["shots"]:
    print(f"=== pascal fold0 shot={shot} ===")
    model = ProtoFormer(shot=shot, reduce_dim=CFG["reduce_dim"],
                        checkpoint_path=SAM2_CKPT).to(device)
    opt = optim.Adam(model.parameters(), lr=CFG["lr"])
    tr = FSSDataset.build_dataloader(CFG["benchmark"], CFG["bsz"], CFG["nworker"], CFG["fold"], "trn", shot=shot)
    va = FSSDataset.build_dataloader(CFG["benchmark"], 1, CFG["nworker"], CFG["fold"], "val", shot=shot)
    hist = {"epoch": [], "trn_loss": [], "trn_miou": [], "val_miou": []}
    best = -1.0
    no_imp = 0
    for e in range(CFG["niter"]):
        tl, tm, tf = train_epoch(e, model, tr, opt, True, amp, scaler)
        hist["epoch"].append(e); hist["trn_loss"].append(float(tl)); hist["trn_miou"].append(float(tm))
        hist["val_miou"].append(None)
        if (e + 1) % CFG.get("val_interval", 3) != 0 and e != CFG["niter"] - 1:
            print(f"epoch {e}: trn_loss={float(tl):.4f} trn_mIoU={float(tm):.2f}")
            continue
        with torch.no_grad():
            vl, vm, vf = train_epoch(e, model, va, opt, False, amp, None)
        hist["val_miou"][-1] = float(vm)
        if float(vm) > best:
            best = float(vm)
            no_imp = 0
            torch.save(model.state_dict(), OUT / f"best_model_{shot}shot.pt")
        else:
            no_imp += 1
            if CFG.get("patience", 20) > 0 and no_imp >= CFG["patience"]:
                print(f"early stop @ epoch {e}")
                break
        print(f"epoch {e}: trn_loss={float(tl):.4f} trn_mIoU={float(tm):.2f} val_mIoU={float(vm):.2f} best={best:.2f}")
        print(f"epoch {e}: trn_loss={float(tl):.4f} trn_mIoU={float(tm):.2f} val_mIoU={float(vm):.2f} best={best:.2f}")
    (OUT / f"history_{shot}shot.json").write_text(json.dumps(hist))
    del model, opt, tr, va
    torch.cuda.empty_cache()


In [ ]:
# 4
from common.logger import AverageMeter

test_hist = {}
for shot in CFG["shots"]:
    print(f"=== pascal fold0 shot={shot} test ===")
    model = ProtoFormer(shot=shot, reduce_dim=CFG["reduce_dim"],
                        checkpoint_path=SAM2_CKPT).to(device)
    model.load_state_dict(torch.load(OUT / f"best_model_{shot}shot.pt", map_location=device, weights_only=False))
    model.eval()
    va = FSSDataset.build_dataloader(CFG["benchmark"], 1, CFG["nworker"], CFG["fold"], "val", shot=shot)
    utils.fix_randseed(0)
    m = AverageMeter(va.dataset)
    for i, b in enumerate(va):
        b = utils.to_cuda(b)
        with torch.no_grad(), amp_context(amp):
            out = model.predict_mask(b)
        pred = (out > 0.5).float().squeeze(1)
        inter, union = Evaluator.classify_prediction(pred.clone(), b)
        m.update(inter, union, b["class_id"], loss=None)
        if i % 100 == 0:
            miou, fb = m.compute_iou()
            print(f"test {i}/{len(va)} mIoU={float(miou):.2f} FB={float(fb):.2f}")
    miou, fb = m.compute_iou()
    test_hist[f"pascal_fold0_{shot}shot"] = {"mIoU": float(miou), "FB_IoU": float(fb), "cfg": CFG}
    del model, va
    torch.cuda.empty_cache()
(OUT / "test_history.json").write_text(json.dumps(test_hist, indent=2))
print(test_hist)


In [ ]:
# 5
files = sorted(p.name for p in OUT.iterdir() if p.is_file())
print(files)
for n in files:
    print(n, (OUT / n).stat().st_size // 1024 // 1024, "MB")
with zipfile.ZipFile(OUT / "protoformer_out.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for n in files:
        if not n.endswith(".zip"):
            z.write(OUT / n, n)
print("saved:", OUT / "protoformer_out.zip")
